# Project 2: Student Performance Prediction
**Level:** Beginner  
**Goal:** Predict whether a student passes based on study time, attendance, previous score, and assignments.

The dataset is **synthetic and illustrative**, generated here for reproducibility. Do not use it to make real decisions about students.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, ConfusionMatrixDisplay,
                             classification_report, RocCurveDisplay)

SEED = 42
rng = np.random.default_rng(SEED)
n = 800
study = rng.uniform(0, 6, n)
attendance = rng.uniform(50, 100, n)
previous = rng.uniform(25, 100, n)
assignments = rng.uniform(0, 100, n)
support = rng.choice(["No", "Yes"], size=n, p=[0.65, 0.35])
logit = -5.5 + 0.48*study + 0.035*attendance + 0.025*previous + 0.018*assignments + 0.45*(support=="Yes")
prob = 1/(1+np.exp(-logit))
passed = rng.binomial(1, prob)
df = pd.DataFrame({"study_hours_daily":study.round(1), "attendance_pct":attendance.round(1),
                   "previous_score":previous.round(1), "assignment_score":assignments.round(1),
                   "extra_support":support, "passed":passed})
display(df.head())
print("Pass rate:", round(df["passed"].mean(), 3))

## 1. Prepare features and split

In [ ]:
X = df.drop(columns="passed")
y = df["passed"]
numeric = ["study_hours_daily", "attendance_pct", "previous_score", "assignment_score"]
categorical = ["extra_support"]
preprocess = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), numeric),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical)
])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

## 2. Train and evaluate classifiers

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=SEED),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=250, min_samples_leaf=3,
                                             random_state=SEED)
}
pipelines, rows = {}, []
for name, estimator in models.items():
    pipe = Pipeline([("preprocess", preprocess), ("model", estimator)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    scores = pipe.predict_proba(X_test)[:, 1]
    pipelines[name] = pipe
    rows.append({"Model":name, "Accuracy":accuracy_score(y_test,pred),
                 "Precision":precision_score(y_test,pred,zero_division=0),
                 "Recall":recall_score(y_test,pred,zero_division=0),
                 "F1":f1_score(y_test,pred,zero_division=0),
                 "ROC_AUC":roc_auc_score(y_test,scores)})
metrics = pd.DataFrame(rows).sort_values("F1", ascending=False)
metrics.round(3)

## 3. Visualize model comparison and confusion matrix

In [ ]:
metrics.set_index("Model")[["Accuracy","Precision","Recall","F1","ROC_AUC"]].plot(
    kind="bar", figsize=(10,5), ylim=(0,1.05)
)
plt.ylabel("Score"); plt.title("Classifier performance on test set")
plt.xticks(rotation=15); plt.tight_layout(); plt.show()

best_name = metrics.iloc[0]["Model"]
best = pipelines[best_name]
ConfusionMatrixDisplay.from_estimator(best, X_test, y_test, cmap=None)
plt.title(f"Confusion matrix — {best_name}")
plt.tight_layout(); plt.show()
print(classification_report(y_test, best.predict(X_test), zero_division=0))

## 4. Example prediction

In [ ]:
new_student = pd.DataFrame([{
    "study_hours_daily": 2.5, "attendance_pct": 88,
    "previous_score": 72, "assignment_score": 80, "extra_support": "Yes"
}])
p_pass = best.predict_proba(new_student)[0, 1]
print("Selected model:", best_name)
print(f"Estimated pass probability on this synthetic example: {p_pass:.1%}")

## 5. Interpretation and limitations
- Precision, recall, F1, and ROC-AUC describe different aspects of classification performance.
- A high score on generated data does not imply performance on real students.
- Real student data requires appropriate permissions, privacy safeguards, and bias analysis.
- Treat predictions as support for human review, never as the sole basis for educational decisions.